In [1]:
import pandas as pd
import numpy as np

In [2]:
import os

PROJECT_ROOT = r"C:\Users\nicol\OneDrive\Bureau\Finance"
os.chdir(PROJECT_ROOT)

print("Current working directory:", os.getcwd())

Current working directory: C:\Users\nicol\OneDrive\Bureau\Finance


In [3]:
df = pd.read_csv("data/interim/market_clean_pred.csv", index_col=0, parse_dates=True)
df.head(10)

,Open,High,Low,Close,Volume,log_return
Date,,,,,,
2024-01-03 00:00:00-05:00,4725.069824,4729.290039,4699.709961,4704.810059,3950760000,-0.008049
2024-01-04 00:00:00-05:00,4697.419922,4726.779785,4687.529785,4688.680176,3715480000,-0.003434
2024-01-05 00:00:00-05:00,4690.569824,4721.490234,4682.109863,4697.240234,3844370000,0.001824
2024-01-08 00:00:00-05:00,4703.700195,4764.540039,4699.819824,4763.540039,3742320000,0.014016
2024-01-09 00:00:00-05:00,4741.930176,4765.470215,4730.350098,4756.500000,3529960000,-0.001479
2024-01-10 00:00:00-05:00,4759.939941,4790.799805,4756.200195,4783.450195,3498680000,0.005650
2024-01-11 00:00:00-05:00,4792.129883,4798.500000,4739.580078,4780.240234,3759890000,-0.000671
2024-01-12 00:00:00-05:00,4791.180176,4802.399902,4768.979980,4783.830078,3486340000,0.000751
2024-01-16 00:00:00-05:00,4772.350098,4782.339844,4747.120117,4765.979980,4260550000,-0.003738


In [4]:
# Volatilité glissante
df["vol_5"]  = df["log_return"].rolling(5).std()
df["vol_10"] = df["log_return"].rolling(10).std()
df["vol_21"] = df["log_return"].rolling(21).std()

In [5]:
# Volatilité exponentielle
df["vol_ewma"] = df["log_return"].ewm(span=21).std()

In [6]:
from src.features.indicators import ATR

# Average True Range (ATR)
df["ATR"] = ATR(df)

In [7]:
import importlib
from src.features import indicators
importlib.reload(indicators)
from src.features.indicators import RSI

# Relative Strength Index (RSI)
df["RSI"] = RSI(df["Close"])

In [8]:
# MACD / Signal MACD
exp1 = df["Close"].ewm(span=12).mean()
exp2 = df["Close"].ewm(span=26).mean()
df["MACD"] = exp1 - exp2
df["MACD_signal"] = df["MACD"].ewm(span=9).mean()

In [9]:
# Volume logarithmique
df["log_volume"] = np.log(df["Volume"])

In [10]:
# Variation du volume
df["volume_change"] = df["log_volume"].diff()

In [11]:
# Volalité réalisée cible
df["target_vol"] = df["log_return"].rolling(21).std().shift(-1)

In [12]:
df_features = df.dropna().copy()
df_features.head(10)

,Open,High,Low,Close,Volume,log_return,vol_5,vol_10,vol_21,vol_ewma,ATR,RSI,MACD,MACD_signal,log_volume,volume_change,target_vol
Date,,,,,,,,,,,,,,,,,
2024-02-01 00:00:00-05:00,4861.109863,4906.970215,4853.520020,4906.189941,4386090000,0.012416,0.010886,0.008163,0.007232,0.008208,40.820033,66.309220,25.375135,21.378787,22.201704,-0.068298,0.007132
2024-02-02 00:00:00-05:00,4916.060059,4975.290039,4907.990234,4958.609863,3974350000,0.010628,0.011727,0.007965,0.007132,0.008254,43.368617,70.091457,29.185241,22.951684,22.103127,-0.098577,0.007122
2024-02-05 00:00:00-05:00,4957.189941,4957.189941,4918.089844,4942.810059,4023640000,-0.003191,0.011613,0.008162,0.007122,0.008025,43.640765,70.423415,30.720756,24.514725,22.115453,0.012326,0.007120
2024-02-06 00:00:00-05:00,4950.160156,4957.770020,4934.879883,4954.229980,4440880000,0.002308,0.011611,0.008155,0.007120,0.007604,41.621477,75.747197,32.335837,26.086369,22.214118,0.098666,0.006756
2024-02-07 00:00:00-05:00,4973.049805,4999.890137,4969.049805,4995.060059,4895590000,0.008208,0.006433,0.008385,0.006756,0.007454,41.555769,75.694816,36.049488,28.086549,22.311601,0.097482,0.006714
2024-02-08 00:00:00-05:00,4995.160156,5000.399902,4987.089844,4997.910156,4341860000,0.000570,0.005647,0.008349,0.006714,0.007101,38.140067,71.919371,38.718030,30.219291,22.191569,-0.120032,0.006716
2024-02-09 00:00:00-05:00,5004.169922,5030.060059,5000.339844,5026.609863,3912990000,0.005726,0.004441,0.008360,0.006716,0.006812,38.393624,73.259904,42.379068,32.657141,22.087568,-0.104001,0.006722
2024-02-12 00:00:00-05:00,5026.830078,5048.390137,5016.830078,5021.839844,3805740000,-0.000949,0.003754,0.008249,0.006722,0.006569,39.068638,71.287748,44.398665,35.009997,22.059776,-0.027791,0.007586
2024-02-13 00:00:00-05:00,4967.939941,4971.299805,4920.310059,4953.169922,4302190000,-0.013769,0.008529,0.009626,0.007586,0.007953,43.529332,59.747962,40.507916,36.111285,22.182390,0.122614,0.007666


In [13]:
df.shape, df_features.shape

((553, 17), (532, 17))

In [14]:
df_features.to_csv("data/processed/features_pred.csv")